# RiskAI – Datenvorverarbeitung (Preprocessing)

Vorbereitung der Daten für das Machine Learning.
Schritte:
1. Laden der Rohdaten
2. Entfernen von Duplikaten
3. Stratifizierter Train/Test-Split
4. Robuste Skalierung der Features `Time` und `Amount` mittels sklearn Pipeline


In [4]:
import pandas as pd 
import numpy as np 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.compose import ColumnTransformer
import joblib 


df = pd.read_csv("../data/raw/creditcard.csv")
print(f"Datensatz geladen {df.shape[0]} Zeilen")

Datensatz geladen 284807 Zeilen


## Duplikate entfernen
wir entfernen die 1.081 doppelten Zeilen, damit unser Modell nicht durch redundante Datenpunkte verzerrt wird oder Data Leakage entsteht.

In [5]:
df_clean = df.drop_duplicates()
print(f"Zeilen vor Bereinigung: {df.shape[0]}")
print(f"Zeilen nach Bereinigung: {df_clean.shape[0]}")
print(f"Entfernte Duplikate: {df.shape[0] - df_clean.shape[0]}")

Zeilen vor Bereinigung: 284807
Zeilen nach Bereinigung: 283726
Entfernte Duplikate: 1081


Hier trennen wir die Zielvariable Class von den restlichen Features und teilen den Datensatz auf.

In [6]:
X = df_clean.drop(columns=["Class"])
y = df_clean["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Trainingstest: {X_train.shape[0]} Zeilen (Betrügsfälle: {y_train.sum()} = {y_train.mean()*100:.3f}%)")
print(f"Testset: {X_test.shape[0]} Zeilen (Betrügsfälle: {y_test.sum()} = {y_test.mean()*100:.3f})")

Trainingstest: 226980 Zeilen (Betrügsfälle: 378 = 0.167%)
Testset: 56746 Zeilen (Betrügsfälle: 95 = 0.167)


Jetzt skalieren wir `Time` und `Amount`. Die Features `V1` bis `V28` sind durch die PCA bereits um 0 zentriert und müssen nicht skaliert werden. Dafür nutzen wir den `ColumnTransformer`

In [7]:
cols_to_scale = ["Time", "Amount"]

preprocessor = ColumnTransformer(
    transformers=[
        ("scaler", RobustScaler(), cols_to_scale)
    ],
    remainder="passthrough"
)

X_train_scaled = preprocessor.fit_transform(X_train)

X_test_scaled = preprocessor.transform(X_test)

neue_spaltenreihenfolge = cols_to_scale + [col for col in X_train.columns if col not in cols_to_scale]
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=neue_spaltenreihenfolge)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=neue_spaltenreihenfolge)

X_train_scaled_df.head()


,Time,Amount,V1,V2,V3,V4,V5,V6,V7,V8,...,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28
0,0.701730,0.137472,2.238954,-1.724499,-2.151484,-2.577803,0.993668,3.565492,-1.785957,0.860122,...,-0.318110,-0.323810,-0.149574,-0.049333,0.278442,0.684735,-0.219028,-0.159167,0.037920,-0.049932
1,-0.048166,-0.209119,-1.315062,1.630783,0.597001,-0.038359,-0.404580,-0.965712,0.212249,0.735381,...,0.392417,-0.067580,-0.238898,-0.946773,0.323904,0.515632,-0.713000,-0.266503,-0.017794,0.051058
2,0.496931,-0.098808,1.908801,0.021184,-2.087997,0.129310,1.161468,0.605244,-0.022371,0.180296,...,-0.994654,-0.210474,0.293609,1.095842,-0.044874,-1.689517,0.106098,0.007758,0.045164,-0.053068
3,0.076666,-0.066242,1.811257,0.316556,0.316751,3.880231,0.048454,1.020163,-0.734868,0.233651,...,-1.693484,-0.228032,0.138869,0.700422,0.174064,0.702997,-0.212523,-0.010018,-0.017740,-0.038006
4,-0.649581,0.026608,1.358817,-1.120881,0.550266,-1.547659,-1.194950,0.275448,-1.201843,0.212889,...,-0.591751,-0.361686,-0.340972,-0.636442,0.252758,-0.344160,-0.064282,-0.439622,0.062524,0.013095


In [ ]:
X_train_scaled_df.to_csv('../data/processed/X_train_scaled.csv', index=False)
X_test_scaled_df.to_csv('../data/processed/X_test_scaled.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)


joblib.dump(preprocessor, '../models/preprocessor.joblib')

print("Vorverarbeitete Daten und Preprocessor wurden erfolgreich gespeichert!")


Vorverarbeitete Daten und Preprocessor wurden erfolgreich gespeichert!
